# Kamień milowy 1

Profil schematu, statystyk technicznych i anomalii technicznych pełnego zrzutu recenzji Steam.

DuckDB wykonuje zapytania SQL w pamięci na plikach Parquet.


## Zbiór źródłowy

Opis kamienia milowego odwołuje się do jednego pliku `steam_reviews.parquet`. Opublikowany zrzut składa się z 663 plików Parquet.

| | |
| --- | --- |
| Lustro na Hugging Face | [adde023/steam-reviews](https://huggingface.co/datasets/adde023/steam-reviews) |
| Deklarowana liczba wierszy | 113 883 717 |
| Rozmiar na dysku | około 23,5 GB, pliki `part.0.parquet` … `part.662.parquet` |
| Zbiór źródłowy na Kaggle | [kieranpoc/steam-reviews](https://www.kaggle.com/datasets/kieranpoc/steam-reviews) |
| Zapis czasu gry | minuty, zgodnie z opisem zbioru na Kaggle; wartości nie są przeliczane |

Nazwy kolumn w pliku różnią się od opisu kamienia milowego. Zapytania zachowują aliasy wyniku i czytają kolumny z pliku.

| Nazwa w opisie kamienia | Kolumna w pliku Parquet |
| --- | --- |
| `review_id` | `recommendationid` |
| `app_id` | `appid` |
| `author.steamid` | `author_steamid` |
| `author.playtime_at_review` | `author_playtime_at_review` |
| `author.playtime_forever` | `author_playtime_forever` |
| `votes_helpful` | `votes_up` |
| `review` | `review` |
| `timestamp_created` | `timestamp_created` |
| `timestamp_updated` | `timestamp_updated` |


In [1]:
from pathlib import Path
import os
import time

import duckdb
import pandas as pd

expectedParquetShards = 663
advertisedRowCount = 113_883_717
float64MantissaBits = 53
float64ExactIntegerLimit = 1 << float64MantissaBits
steamIdRangeUlp = 16
float64Exponent56 = 1 << 56
float64Exponent57 = 1 << 57
counterOverflowThreshold = 100_000
exampleReviewId = 148919350
viewerPrintedSteamId = "76561198826729330"
duckdbMemoryLimit = "16GB"
duckdbThreadCount = os.cpu_count()

sourceColumns = {
    "reviewId": "recommendationid",
    "appId": "appid",
    "authorSteamId": "author_steamid",
    "playtimeAtReview": "author_playtime_at_review",
    "playtimeForever": "author_playtime_forever",
    "votesHelpful": "votes_up",
    "reviewText": "review",
    "timestampCreated": "timestamp_created",
    "timestampUpdated": "timestamp_updated",
}

projectRoot = Path.cwd().resolve()
if projectRoot.name == "notebooks":
    projectRoot = projectRoot.parent
rawDirectory = projectRoot / "data" / "raw" / "steam-reviews"
rawParquetGlob = (rawDirectory / "*.parquet").as_posix()

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print(f"duckdb {duckdb.__version__}")
print(f"project root: {projectRoot}")
print(f"parquet glob: {rawParquetGlob}")


duckdb 2.0.0
project root: D:\Projects\game-lifecycle-decay-analitics
parquet glob: D:/Projects/game-lifecycle-decay-analitics/data/raw/steam-reviews/*.parquet


## Plik na dysku

Pobranie plików Parquet następuje, gdy zestaw na dysku jest niekompletny. Katalog `data/raw/` jest wyłączony z repozytorium.


In [3]:
shardPaths = sorted(rawDirectory.glob("*.parquet"))
if len(shardPaths) != expectedParquetShards:
    from huggingface_hub import snapshot_download

    rawDirectory.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id="adde023/steam-reviews",
        repo_type="dataset",
        allow_patterns=["*.parquet"],
        local_dir=str(rawDirectory),
        max_workers=duckdbThreadCount,
    )
    shardPaths = sorted(rawDirectory.glob("*.parquet"))

if len(shardPaths) != expectedParquetShards:
    raise RuntimeError(
        f"Expected {expectedParquetShards} parquet shards, found {len(shardPaths)} in {rawDirectory}"
    )

rawBytes = sum(path.stat().st_size for path in shardPaths)
print(f"shards: {len(shardPaths)}")
print(f"bytes: {rawBytes}")
print(f"GiB: {rawBytes / (1024 ** 3):.2f}")


shards: 663
bytes: 23463649848
GiB: 21.85


In [4]:
connection = duckdb.connect()
connection.execute(f"SET memory_limit = '{duckdbMemoryLimit}'")
connection.execute(f"SET threads = {duckdbThreadCount}")
connection.execute("SET preserve_insertion_order = false")
print(connection.execute("SELECT current_setting('memory_limit'), current_setting('threads')").fetchall())


[('14.9 GiB', 12)]


## 1. Schemat (`DESCRIBE`)

`DESCRIBE` czyta metadane Parquet pełnego zrzutu i nie skanuje 113 milionów wierszy.

Typem odpowiednim dla 64-bitowego identyfikatora Steam jest `BIGINT`, `HUGEINT` albo napis. `DOUBLE` traci dokładność powyżej 2^53.


In [5]:
describeSql = f"DESCRIBE SELECT * FROM read_parquet('{rawParquetGlob}')"
print(describeSql)
schemaFrame = connection.execute(describeSql).fetchdf()
schemaFrame


DESCRIBE SELECT * FROM read_parquet('D:/Projects/game-lifecycle-decay-analitics/data/raw/steam-reviews/*.parquet')


,column_name,column_type,null,key,default,extra
0,recommendationid,DOUBLE,YES,None,None,None
1,appid,DOUBLE,YES,None,None,None
2,game,VARCHAR,YES,None,None,None
3,author_steamid,DOUBLE,YES,None,None,None
4,author_num_games_owned,DOUBLE,YES,None,None,None
5,author_num_reviews,DOUBLE,YES,None,None,None
6,author_playtime_forever,DOUBLE,YES,None,None,None
7,author_playtime_last_two_weeks,DOUBLE,YES,None,None,None
8,author_playtime_at_review,DOUBLE,YES,None,None,None
9,author_last_played,DOUBLE,YES,None,None,None


In [6]:
schemaByName = dict(zip(schemaFrame["column_name"], schemaFrame["column_type"]))
requiredColumns = sorted(sourceColumns.values())
missingColumns = [name for name in requiredColumns if name not in schemaByName]
if missingColumns:
    raise RuntimeError(f"W schemacie brakuje kolumn: {missingColumns}")

safeIntegerTypes = {"BIGINT", "HUGEINT", "UBIGINT", "INTEGER", "UINTEGER", "VARCHAR"}
for columnName in (sourceColumns["reviewId"], sourceColumns["authorSteamId"]):
    columnType = schemaByName[columnName]
    print(f"{columnName}: {columnType}")
    if columnType not in safeIntegerTypes:
        print(f"  {columnName} nie jest kolumną całkowitą 64-bitową")

print(f"__null_dask_index__: {schemaByName.get('__null_dask_index__', 'absent')}")


recommendationid: DOUBLE
  recommendationid nie jest kolumną całkowitą 64-bitową
author_steamid: DOUBLE
  author_steamid nie jest kolumną całkowitą 64-bitową
__null_dask_index__: BIGINT


### Czy zapisane identyfikatory są już zaokrąglone?

`steamIdRangeUlp` wynosi 16, ponieważ każdy SteamID64 leży w przedziale `[2^56, 2^57)`, a odstęp float64 w tym zakresie wynosi 16. Przy wiernym zapisie liczby całkowitej wiele identyfikatorów nie spełnia warunku `value % 16 = 0`. Gdy każda zapisana wartość dzieli się przez 16, młodsze bity zostały odrzucone przy zapisie pliku.

Recenzja `148919350` jest wierszem wskazanym w podglądzie zbioru.


In [7]:
identifierSql = f"""
SELECT
    count_if({sourceColumns["authorSteamId"]} IS NOT NULL) AS non_null_steamids,
    count_if(
        CAST({sourceColumns["authorSteamId"]} AS HUGEINT) % {steamIdRangeUlp} <> 0
    ) AS steamids_not_multiple_of_ulp,
    count_if(
        {sourceColumns["authorSteamId"]} < {float64Exponent56}
        OR {sourceColumns["authorSteamId"]} >= {float64Exponent57}
    ) AS steamids_outside_ulp16_range,
    printf('%.0f', MIN({sourceColumns["authorSteamId"]})) AS min_steamid,
    printf('%.0f', MAX({sourceColumns["authorSteamId"]})) AS max_steamid,
    MIN({sourceColumns["reviewId"]}) AS min_review_id,
    MAX({sourceColumns["reviewId"]}) AS max_review_id,
    count_if({sourceColumns["reviewId"]} > {float64ExactIntegerLimit}) AS review_ids_beyond_exact_float64,
    count_if(
        {sourceColumns["reviewId"]} <> CAST(CAST({sourceColumns["reviewId"]} AS BIGINT) AS DOUBLE)
    ) AS review_ids_not_integer
FROM read_parquet('{rawParquetGlob}')
"""
print(identifierSql)
%time identifierFrame = connection.execute(identifierSql).fetchdf()
identifierFrame



SELECT
    count_if(author_steamid IS NOT NULL) AS non_null_steamids,
    count_if(
        CAST(author_steamid AS HUGEINT) % 16 <> 0
    ) AS steamids_not_multiple_of_ulp,
    count_if(
        author_steamid < 72057594037927936
        OR author_steamid >= 144115188075855872
    ) AS steamids_outside_ulp16_range,
    printf('%.0f', MIN(author_steamid)) AS min_steamid,
    printf('%.0f', MAX(author_steamid)) AS max_steamid,
    MIN(recommendationid) AS min_review_id,
    MAX(recommendationid) AS max_review_id,
    count_if(recommendationid > 9007199254740992) AS review_ids_beyond_exact_float64,
    count_if(
        recommendationid <> CAST(CAST(recommendationid AS BIGINT) AS DOUBLE)
    ) AS review_ids_not_integer
FROM read_parquet('D:/Projects/game-lifecycle-decay-analitics/data/raw/steam-reviews/*.parquet')



CPU times: total: 58.8 s
Wall time: 6.09 s


,non_null_steamids,steamids_not_multiple_of_ulp,steamids_outside_ulp16_range,min_steamid,max_steamid,min_review_id,max_review_id,review_ids_beyond_exact_float64,review_ids_not_integer
0,113883717.0,0.0,0.0,76561197960265728,76561199568568192,43.0,149450036.0,0.0,0.0


In [8]:
exampleSql = f"""
SELECT
    printf('%.0f', {sourceColumns["reviewId"]}) AS review_id,
    printf('%.0f', {sourceColumns["authorSteamId"]}) AS stored_steamid,
    CAST({sourceColumns["authorSteamId"]} AS HUGEINT) % {steamIdRangeUlp} AS remainder_mod_ulp
FROM read_parquet('{rawParquetGlob}')
WHERE {sourceColumns["reviewId"]} = {exampleReviewId}
LIMIT 5
"""
print(exampleSql)
%time exampleFrame = connection.execute(exampleSql).fetchdf()
storedSteamId = exampleFrame.loc[0, "stored_steamid"]
print(f"viewer printed: {viewerPrintedSteamId}")
print(f"parquet stores: {storedSteamId}")
print(f"absolute difference: {abs(int(viewerPrintedSteamId) - int(storedSteamId))}")
exampleFrame



SELECT
    printf('%.0f', recommendationid) AS review_id,
    printf('%.0f', author_steamid) AS stored_steamid,
    CAST(author_steamid AS HUGEINT) % 16 AS remainder_mod_ulp
FROM read_parquet('D:/Projects/game-lifecycle-decay-analitics/data/raw/steam-reviews/*.parquet')
WHERE recommendationid = 148919350
LIMIT 5



CPU times: total: 4.89 s
Wall time: 604 ms
viewer printed: 76561198826729330
parquet stores: 76561198826729328
absolute difference: 2


,review_id,stored_steamid,remainder_mod_ulp
0,148919350,76561198826729328,0.0


In [9]:
steamIdsOutsideRange = int(identifierFrame.loc[0, "steamids_outside_ulp16_range"])
steamIdsNotMultiple = int(identifierFrame.loc[0, "steamids_not_multiple_of_ulp"])
reviewIdsBeyondExact = int(identifierFrame.loc[0, "review_ids_beyond_exact_float64"])
reviewIdsNotInteger = int(identifierFrame.loc[0, "review_ids_not_integer"])

if steamIdsOutsideRange != 0:
    raise RuntimeError("Część identyfikatorów steamid leży poza pasmem ULP 16. Test modulo wymaga innej szerokości przerwy.")
if steamIdsNotMultiple != 0:
    print("Część zapisanych wartości author_steamid nie jest wielokrotnością 16. Zaokrąglenie nie jest jednolite.")
else:
    print("Każda zapisana wartość author_steamid jest wielokrotnością 16. Młodsze 4 bity zostały utracone.")

if reviewIdsBeyondExact == 0 and reviewIdsNotInteger == 0:
    print("Każda wartość recommendationid mieści się w mantysie float64. Wartości są dokładne, a typ kolumny pozostaje DOUBLE.")
else:
    print("Część wartości recommendationid jest zaokrąglona albo przekracza 2^53.")


Każda zapisana wartość author_steamid jest wielokrotnością 16. Młodsze 4 bity zostały utracone.
Każda wartość recommendationid mieści się w mantysie float64. Wartości są dokładne, a typ kolumny pozostaje DOUBLE.


### Wynik

| Kontrola | Wynik |
| --- | --- |
| `recommendationid` | `DOUBLE`. Wartości od 43 do 149 450 036. Każda wartość pozostaje dokładną liczbą całkowitą. |
| `author_steamid` | `DOUBLE`. Wszystkie 113 883 717 zapisanych wartości są wielokrotnością 16, więc młodsze 4 bity zostały utracone. |
| Przykład | Recenzja `148919350` jest zapisana jako `76561198826729328`. Podgląd zbioru pokazuje `76561198826729330`. |
| Różne konta | 27 975 166 zapisanych wartości float. Liczba jest dolnym ograniczeniem liczby kont. |


## 2. Statystyki i braki danych

Jedna agregacja pełnego zrzutu: liczba wierszy, liczba gier, liczba zapisanych kont, braki w `review` i `author_steamid` oraz `MIN` / `MAX` / `AVG` dla obu kolumn czasu gry i głosów pomocnych.

`COUNT(DISTINCT)` oraz `AVG` pomijają wartości puste. `unique_authors` liczy różne zapisane wartości float.


In [10]:
hardStatsSql = f"""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT {sourceColumns["appId"]}) AS unique_games,
    COUNT(DISTINCT {sourceColumns["authorSteamId"]}) AS unique_authors,
    count_if({sourceColumns["reviewText"]} IS NULL) AS null_reviews,
    count_if({sourceColumns["authorSteamId"]} IS NULL) AS null_authors,
    MIN({sourceColumns["playtimeAtReview"]}) AS min_playtime_at_review,
    MAX({sourceColumns["playtimeAtReview"]}) AS max_playtime_at_review,
    AVG({sourceColumns["playtimeAtReview"]}) AS avg_playtime_at_review,
    MIN({sourceColumns["playtimeForever"]}) AS min_playtime_forever,
    MAX({sourceColumns["playtimeForever"]}) AS max_playtime_forever,
    AVG({sourceColumns["playtimeForever"]}) AS avg_playtime_forever,
    MIN({sourceColumns["votesHelpful"]}) AS min_votes_helpful,
    MAX({sourceColumns["votesHelpful"]}) AS max_votes_helpful,
    AVG({sourceColumns["votesHelpful"]}) AS avg_votes_helpful
FROM read_parquet('{rawParquetGlob}')
"""
print(hardStatsSql)
startedAt = time.perf_counter()
%time hardStatsFrame = connection.execute(hardStatsSql).fetchdf()
hardStatsElapsedSeconds = time.perf_counter() - startedAt
integerStatColumns = [
    "total_records",
    "unique_games",
    "unique_authors",
    "null_reviews",
    "null_authors",
]
for columnName in integerStatColumns:
    hardStatsFrame[columnName] = hardStatsFrame[columnName].astype("int64")
print(f"elapsed_seconds: {hardStatsElapsedSeconds:.3f}")
hardStatsFrame



SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT appid) AS unique_games,
    COUNT(DISTINCT author_steamid) AS unique_authors,
    count_if(review IS NULL) AS null_reviews,
    count_if(author_steamid IS NULL) AS null_authors,
    MIN(author_playtime_at_review) AS min_playtime_at_review,
    MAX(author_playtime_at_review) AS max_playtime_at_review,
    AVG(author_playtime_at_review) AS avg_playtime_at_review,
    MIN(author_playtime_forever) AS min_playtime_forever,
    MAX(author_playtime_forever) AS max_playtime_forever,
    AVG(author_playtime_forever) AS avg_playtime_forever,
    MIN(votes_up) AS min_votes_helpful,
    MAX(votes_up) AS max_votes_helpful,
    AVG(votes_up) AS avg_votes_helpful
FROM read_parquet('D:/Projects/game-lifecycle-decay-analitics/data/raw/steam-reviews/*.parquet')



CPU times: total: 8min 39s
Wall time: 2min 57s
elapsed_seconds: 177.152


,total_records,unique_games,unique_authors,null_reviews,null_authors,min_playtime_at_review,max_playtime_at_review,avg_playtime_at_review,min_playtime_forever,max_playtime_forever,avg_playtime_forever,min_votes_helpful,max_votes_helpful,avg_votes_helpful
0,113883717,105893,27975166,1532,0,0.0,4880175.0,7092.719567,0.0,6007985.0,15394.045024,0.0,62199.0,2.306077


In [11]:
totalRecords = int(hardStatsFrame.loc[0, "total_records"])
print(f"advertised rows: {advertisedRowCount}")
print(f"counted rows:    {totalRecords}")
print(f"difference:      {totalRecords - advertisedRowCount}")


advertised rows: 113883717
counted rows:    113883717
difference:      0


### Wynik

Czas ścienny: 2 min 25 s.

| Miara | Wartość |
| --- | ---: |
| `total_records` | 113 883 717 |
| `unique_games` | 105 893 |
| `unique_authors` | 27 975 166 |
| `null_reviews` | 1 532 |
| `null_authors` | 0 |

Czas gry jest zapisany w minutach.

| Pole | Minimum | Maksimum | Średnia |
| --- | ---: | ---: | ---: |
| Czas gry przy recenzji | 0 | 4 880 175 | 7 092,72 |
| Czas gry łącznie | 0 | 6 007 985 | 15 394,05 |
| Głosy pomocne | 0 | 62 199 | 2,31 |

`COUNT(*)` jest zgodne z deklarowaną liczbą wierszy 113 883 717.


## 3. Anomalie techniczne

`count_if` liczy wiersze pełnego zrzutu, w których warunek jest prawdziwy. Porównanie z `NULL` daje wartość nieznaną, więc taki wiersz nie wchodzi do licznika.

| Alias | Warunek |
| --- | --- |
| `playtime_teleportation` | czas gry przy recenzji jest większy od łącznego czasu gry |
| `non_positive_playtime` | czas gry przy recenzji wynosi zero albo jest ujemny |
| `clock_skew` | `timestamp_updated` jest wcześniejszy niż `timestamp_created` |
| `counter_overflow` | głosy pomocne powyżej 100 000 |


In [12]:
technicalAnomalySql = f"""
SELECT
    count_if(
        {sourceColumns["playtimeAtReview"]} > {sourceColumns["playtimeForever"]}
    ) AS playtime_teleportation,
    count_if({sourceColumns["playtimeAtReview"]} <= 0) AS non_positive_playtime,
    count_if(
        {sourceColumns["timestampUpdated"]} < {sourceColumns["timestampCreated"]}
    ) AS clock_skew,
    count_if({sourceColumns["votesHelpful"]} > {counterOverflowThreshold}) AS counter_overflow
FROM read_parquet('{rawParquetGlob}')
"""
print(technicalAnomalySql)
startedAt = time.perf_counter()
%time technicalAnomalyFrame = connection.execute(technicalAnomalySql).fetchdf()
technicalAnomalyElapsedSeconds = time.perf_counter() - startedAt
for columnName in technicalAnomalyFrame.columns:
    technicalAnomalyFrame[columnName] = technicalAnomalyFrame[columnName].astype("int64")
print(f"elapsed_seconds: {technicalAnomalyElapsedSeconds:.3f}")
technicalAnomalyFrame



SELECT
    count_if(
        author_playtime_at_review > author_playtime_forever
    ) AS playtime_teleportation,
    count_if(author_playtime_at_review <= 0) AS non_positive_playtime,
    count_if(
        timestamp_updated < timestamp_created
    ) AS clock_skew,
    count_if(votes_up > 100000) AS counter_overflow
FROM read_parquet('D:/Projects/game-lifecycle-decay-analitics/data/raw/steam-reviews/*.parquet')



CPU times: total: 16.3 s
Wall time: 5.96 s
elapsed_seconds: 5.959


,playtime_teleportation,non_positive_playtime,clock_skew,counter_overflow
0,45078,1871268,2,0


### Wynik

Czas ścienny: 6,6 s.

| Anomalia | Wiersze |
| --- | ---: |
| `playtime_teleportation` | 45 078 |
| `non_positive_playtime` | 1 871 268 |
| `clock_skew` | 2 |
| `counter_overflow` | 0 |

Największa liczba głosów pomocnych w pełnym zrzucie wynosi 62 199.


## 4. Podsumowanie

| Miara | Wartość |
| --- | ---: |
| `total_records` | 113 883 717 |
| `unique_games` | 105 893 |
| `unique_authors` | 27 975 166 |
| `null_reviews` | 1 532 |
| `playtime_teleportation` | 45 078 |
| `non_positive_playtime` | 1 871 268 |
| `clock_skew` | 2 |
| `counter_overflow` | 0 |

Każda zapisana wartość `author_steamid` jest wielokrotnością 16, więc liczba kont jest dolnym ograniczeniem. `COUNT(*)` jest zgodne z deklarowaną liczbą wierszy. Największa liczba głosów pomocnych wynosi 62 199.
